# Chapter 23: Correlation, Association, and Regression

**Level:** Applied  
**Objectives:** visualize paired data; compare Pearson and Spearman correlation; fit and diagnose simple regression; examine outlier sensitivity.  
**Prerequisites:** Chapters 15 to 17 and 21 to 22.  
**Estimated study time:** 75 minutes.

The data below are synthetic NRG route observations generated with seed 20260829. No external or private data are used.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(20260829)
distance = np.linspace(5, 40, 40)
congestion = rng.integers(1, 6, size=40)
delivery_time = 24 + 1.45 * distance + 3.2 * congestion + rng.normal(0, 4, 40)
print(f"Rows: {len(distance)}")
print(f"Distance range: {distance.min():.1f} to {distance.max():.1f} km")

Rows: 40
Distance range: 5.0 to 40.0 km


In [2]:
pearson = np.corrcoef(distance, delivery_time)[0, 1]
rank_x = np.argsort(np.argsort(distance)) + 1
rank_y = np.argsort(np.argsort(delivery_time)) + 1
spearman = np.corrcoef(rank_x, rank_y)[0, 1]
print(f"Pearson r: {pearson:.3f}")
print(f"Spearman rho: {spearman:.3f}")

Pearson r: 0.933
Spearman rho: 0.929


In [3]:
slope, intercept = np.polyfit(distance, delivery_time, 1)
fitted = intercept + slope * distance
residuals = delivery_time - fitted
sse = np.sum(residuals ** 2)
sst = np.sum((delivery_time - delivery_time.mean()) ** 2)
r_squared = 1 - sse / sst
rmse = np.sqrt(np.mean(residuals ** 2))
print(f"Intercept: {intercept:.2f} minutes")
print(f"Slope: {slope:.2f} minutes per km")
print(f"R-squared: {r_squared:.3f}")
print(f"RMSE: {rmse:.2f} minutes")

Intercept: 37.33 minutes
Slope: 1.40 minutes per km
R-squared: 0.871
RMSE: 5.57 minutes


In [4]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(distance, delivery_time, c=congestion, cmap="viridis", edgecolor="black")
axes[0].plot(distance, fitted, color="firebrick", label="OLS line")
axes[0].set(xlabel="Distance (km)", ylabel="Delivery time (minutes)", title="NRG route observations")
axes[0].legend()
axes[1].scatter(fitted, residuals, color="steelblue", edgecolor="black")
axes[1].axhline(0, color="firebrick", linewidth=1)
axes[1].set(xlabel="Fitted time", ylabel="Residual", title="Residual diagnostic")
fig.tight_layout()
plt.show()

<Figure size 1100x400 with 2 Axes>

In [5]:
x_curve = np.arange(1, 11)
y_curve = x_curve ** 2
curve_pearson = np.corrcoef(x_curve, y_curve)[0, 1]
curve_spearman = np.corrcoef(np.argsort(np.argsort(x_curve)), np.argsort(np.argsort(y_curve)))[0, 1]
distance_outlier = np.append(distance, 65)
time_outlier = np.append(delivery_time, 35)
outlier_r = np.corrcoef(distance_outlier, time_outlier)[0, 1]
print(f"Curved pattern Pearson: {curve_pearson:.3f}")
print(f"Curved pattern Spearman: {curve_spearman:.3f}")
print(f"Pearson before outlier: {pearson:.3f}")
print(f"Pearson after outlier: {outlier_r:.3f}")

Curved pattern Pearson: 0.975
Curved pattern Spearman: 1.000
Pearson before outlier: 0.933
Pearson after outlier: 0.570


In [6]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(x_curve, y_curve, color="darkorange")
axes[0].set(xlabel="Congestion index", ylabel="Delay score", title="Strong monotonic, curved pattern")
axes[1].scatter(distance_outlier, time_outlier, color="steelblue")
axes[1].scatter([65], [35], color="firebrick", s=80, label="High-leverage point")
axes[1].set(xlabel="Distance (km)", ylabel="Delivery time", title="Correlation sensitivity")
axes[1].legend()
fig.tight_layout()
plt.show()

<Figure size 1100x400 with 2 Axes>

## Interpretation and limitations

Distance has a strong positive linear association with delivery time in this synthetic sample. The regression slope is an association conditional only on the chosen one-predictor model. Congestion is an omitted route characteristic, so the slope should not be presented as a causal effect. The high-leverage example shows why a plot and sensitivity analysis belong beside a coefficient. Predictive claims require validation on later routes.

## Practice

Add route type as colours, calculate correlations within each type, and compare them with the pooled correlation. Then explain any difference. The cell below is intentionally blank for learner work.